# Predicting Student Health Risk - Robust LightGBM Baseline with Feature Engineering

This notebook implements an end-to-end Machine Learning pipeline using **LightGBM** to predict student health conditions (`fit`, `unhealthy`, `at-risk`) for the Kaggle Playground Series (Season 6, Episode 7). 

The evaluation metric for this competition is **Balanced Accuracy**, which requires careful handling of class imbalances and stable validation strategies.

---

## Pipeline Architecture

### 1. Feature Engineering (Domain-Specific Metrics)
Since the dataset is synthetically generated, capturing hidden mathematical interactions between physiological variables yields significant performance boosts. We extract:
*   `calorie_per_minute`: Efficiency of energy expenditure during exercise.
*   `steps_per_calorie`: Movement density relative to energy burned.
*   `sleep_efficiency`: Interaction between sleep duration and mapped categorical sleep quality.
*   `heart_rate_to_bmi`: Physical strain index matching cardiac activity against body mass index.

### 2. Categorical Encoding & Missing Data
*   Categorical features are strictly cast to pandas `category` types, allowing LightGBM to natively split on them via its optimal partitioning algorithm without inflating dimensionality (unlike One-Hot Encoding).
*   Missing values (`NaN`s) are left untouched, leveraging LightGBM's inherent capability to learn the optimal default branching direction for missing values.

### 3. Validation Strategy (5-Fold Stratified K-Fold)
*   We utilize a **StratifiedKFold** validation scheme with 5 splits to ensure that the distribution of target health conditions remains identical across all training and validation folds.
*   This setup prevents target leakage and yields a highly reliable Out-of-Fold (OOF) score that closely tracks the Public/Private Leaderboard.

### 4. Hyperparameter Selection & Regularization
The `LGBMClassifier` is configured with robust regularization parameters to suppress overfitting on synthetic patterns:
*   `learning_rate=0.03` & `n_estimators=1800`: Slower, more precise gradient descent steps.
*   `max_depth=7` & `num_leaves=63`: Restricted tree complexity to prevent deep memorization.
*   `subsample=0.8` & `colsample_bytree=0.8`: Feature and row bagging to increase structural diversity.
*   `class_weight='balanced'`: Directly aligns model optimization with the competition’s **Balanced Accuracy** metric.
*   `early_stopping(stopping_rounds=50)`: Dynamically halts training per fold if the validation multi-logloss plateaus.

### 5. Multi-Class Inference & Ensembling
Predictions across all 5 distinct models are averaged in their probabilistic form (`predict_proba`) via **Soft Voting**. Taking the `argmax` of these blended probabilities ensures the final categorical submission is highly stable and generalized.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from lightgbm import early_stopping, log_evaluation

import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv(r"C:\Users\Afaq_Lite\Downloads\train.csv")
train

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690083,690083,at-risk,6.31,69.7,24.11,2157.0,NaN,30.8,3.00,non-veg,high,poor,active,yes,female
690084,690084,at-risk,5.78,54.0,26.36,2858.0,6488.0,52.4,1.46,veg,medium,average,moderate,no,male
690085,690085,fit,7.64,85.7,21.91,2195.0,9241.0,41.3,1.57,non-veg,NaN,average,active,no,male
690086,690086,at-risk,6.74,73.0,18.73,2061.0,13366.0,56.6,2.60,balanced,NaN,average,active,yes,male


In [3]:
test = pd.read_csv(r"C:\Users\Afaq_Lite\Downloads\test.csv")
test


,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295748,985836,7.13,77.5,22.20,2221.0,13981.0,65.6,2.17,balanced,low,average,active,occasional,other
295749,985837,6.25,69.5,20.93,2242.0,2370.0,23.6,1.98,balanced,NaN,average,moderate,yes,female
295750,985838,NaN,77.0,23.62,2111.0,7647.0,25.1,2.18,veg,high,average,sedentary,no,other
295751,985839,7.25,66.3,21.47,2009.0,4672.0,25.1,2.02,veg,high,poor,moderate,no,female


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  object 
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  object 
 10  stress_level             607277 non-null  object 
 11  sleep_quality            631757 non-null  object 
 12  physical_activity_level  653467 non-null  object 
 13  smoking_alcohol          661506 non-null  object 
 14  gend

In [5]:
train.isna().sum()

id                             0
health_condition               0
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64

In [6]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       295753 non-null  int64  
 1   sleep_duration           263182 non-null  float64
 2   heart_rate               292396 non-null  float64
 3   bmi                      289797 non-null  float64
 4   calorie_expenditure      273101 non-null  float64
 5   step_count               289789 non-null  float64
 6   exercise_duration        292795 non-null  float64
 7   water_intake             277120 non-null  float64
 8   diet_type                292795 non-null  object 
 9   stress_level             260263 non-null  object 
 10  sleep_quality            270754 non-null  object 
 11  physical_activity_level  280058 non-null  object 
 12  smoking_alcohol          283504 non-null  object 
 13  gender                   286593 non-null  object 
dtypes: f

In [7]:
test.isna().sum()

id                             0
sleep_duration             32571
heart_rate                  3357
bmi                         5956
calorie_expenditure        22652
step_count                  5964
exercise_duration           2958
water_intake               18633
diet_type                   2958
stress_level               35490
sleep_quality              24999
physical_activity_level    15695
smoking_alcohol            12249
gender                      9160
dtype: int64

In [8]:
submission = pd.DataFrame({"id": test["id"]})


In [9]:
train.duplicated().sum()

np.int64(0)

In [10]:
test.duplicated().sum()

np.int64(0)

In [11]:
target_col = "health_condition"
features = [col for col in train.columns if col not in ["id", target_col]]


In [12]:
categorical_cols = ["diet_type", "stress_level", "sleep_quality", 
                    "physical_activity_level", "smoking_alcohol", "gender"]

In [13]:
for df in [train, test]:
    df['calorie_per_minute'] = df['calorie_expenditure'] / (df['exercise_duration'] + 1)
    df['steps_per_calorie'] = df['step_count'] / (df['calorie_expenditure'] + 1)

    sleep_map = {'Poor': 1, 'Fair': 2, 'Good': 3, 'Excellent': 4} 
    df['sleep_quality_num'] = df['sleep_quality'].map(sleep_map).fillna(2)
    df['sleep_efficiency'] = df['sleep_duration'] * df['sleep_quality_num']

    df['heart_rate_to_bmi'] = df['heart_rate'] / (df['bmi'] + 1)

In [ ]:
features = [col for col in train.columns if col not in ["id", target_col]]

In [14]:
le_target = LabelEncoder()
train[target_col] = le_target.fit_transform(train[target_col])

In [15]:
for col in categorical_cols:
    train[col] = train[col].astype(str).astype('category')
    test[col] = test[col].astype(str).astype('category')

In [16]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)

oof_preds = np.zeros((len(train), 3))
test_preds = np.zeros((len(test), 3))

In [17]:
for fold, (train_idx, val_idx) in enumerate(skf.split(train, train[target_col])):
    print(f"\n**** {fold + 1} ---")
    
    X_train, y_train = train[features].iloc[train_idx], train[target_col].iloc[train_idx]
    X_val, y_val = train[features].iloc[val_idx], train[target_col].iloc[val_idx]
 
    model = LGBMClassifier(
        objective='multiclass',
        num_class=3,
        metric='multi_logloss',
        boosting_type='gbdt',
        learning_rate=0.03,   
        n_estimators=1800,       
        max_depth=7,             
        num_leaves=63,           
        subsample=0.8,            
        colsample_bytree=0.8,    
        random_state=42,
        class_weight='balanced',
        n_jobs=-1 )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(period=200) ])
    
    oof_preds[val_idx] = model.predict_proba(X_val)
    test_preds += model.predict_proba(test[features]) / n_splits 
    
    val_pred_class = np.argmax(oof_preds[val_idx], axis=1)
    fold_score = balanced_accuracy_score(y_val, val_pred_class)
    print(f" {fold + 1} (Balanced Accuracy): {fold_score:.4f}")

overall_score = balanced_accuracy_score(train[target_col], np.argmax(oof_preds, axis=1))
print(f"\n the final score: {overall_score:.4f}")


**** 1 ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015907 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1813
[LightGBM] [Info] Number of data points in the train set: 552070, number of used features: 13
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[200]	valid_0's multi_logloss: 0.187372
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

In [20]:
test_classes = np.argmax(test_preds, axis=1)
submission[target_col] = le_target.inverse_transform(test_classes)


submission.to_csv("final_hypertuned_submission.csv", index=False)
print('Successfully saved:final_hypertuned_submission.csv')

Successfully saved:final_hypertuned_submission.csv
